# Multi-Layer Perceptron & Backpropagation from Scratch

## Why do we need more than one neuron?

In the previous notebook we saw that a single **perceptron** can learn any decision boundary that is a straight line. But many real-world problems are **not** linearly separable — no single line (or hyperplane) can split the classes.

A **multi-layer perceptron (MLP)** solves this by stacking neurons into layers and adding nonlinear **activation functions** between them. This lets the network learn curved, complex decision boundaries.

To train an MLP we need **backpropagation** — an algorithm that efficiently computes how much each weight contributed to the overall error, so we can update them all with gradient descent.

---

This notebook:
- generates a non-linearly separable 2D dataset (concentric circles)
- shows why a single perceptron fails on it
- builds an MLP from scratch with NumPy
- derives and implements backpropagation step by step
- visualizes how the decision boundary evolves during training
- experiments with learning rate and network size

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.style.use("seaborn-v0_8-whitegrid")

## The dataset — concentric circles

We generate two classes arranged in concentric rings. The inner ring is one class, the outer ring is the other. No single straight line can separate them — this is the kind of problem that requires a nonlinear model.

In [ ]:
def make_circles(n_samples=300, noise=0.08, factor=0.4, seed=42):
    """Generate two concentric circles of 2D points.

    Parameters
    ----------
    n_samples : int
        Total number of points (split evenly between inner and outer ring).
    noise : float
        Standard deviation of Gaussian noise added to the coordinates.
    factor : float
        Radius ratio of inner to outer ring (0 < factor < 1).
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    X : ndarray of shape (n_samples, 2)
    y : ndarray of shape (n_samples,) with values 0 (outer) / 1 (inner)
    """
    rng = np.random.default_rng(seed)
    n_outer = n_samples // 2
    n_inner = n_samples - n_outer

    # Angles uniformly spaced around the circle.
    angles_outer = np.linspace(0, 2 * np.pi, n_outer, endpoint=False)
    angles_inner = np.linspace(0, 2 * np.pi, n_inner, endpoint=False)

    # Outer ring at radius 1, inner ring at radius `factor`.
    outer_x = np.cos(angles_outer)
    outer_y = np.sin(angles_outer)
    inner_x = factor * np.cos(angles_inner)
    inner_y = factor * np.sin(angles_inner)

    X = np.vstack([
        np.column_stack([outer_x, outer_y]),
        np.column_stack([inner_x, inner_y]),
    ])
    y = np.concatenate([np.zeros(n_outer), np.ones(n_inner)])

    # Add noise and shuffle.
    X += rng.normal(0, noise, X.shape)
    order = rng.permutation(n_samples)
    X, y = X[order], y[order]

    return X, y


X, y = make_circles(n_samples=300, seed=42)
print(f"Dataset: {X.shape[0]} points, 2 classes")
print(f"  Class 0 (outer ring): {(y == 0).sum()} points")
print(f"  Class 1 (inner ring): {(y == 1).sum()} points")

In [ ]:
def plot_data(X, y, ax=None, title=""):
    """Scatter plot of the two-class dataset."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=40, color="tab:orange",
               edgecolors="k", linewidths=0.3, label="class 0 (outer)")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=40, color="tab:blue",
               edgecolors="k", linewidths=0.3, label="class 1 (inner)")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_aspect("equal")
    ax.legend()
    if title:
        ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: the dataset.
plot_data(X, y, ax=axes[0], title="Concentric circles dataset")

# Right: best linear attempt — train a simple perceptron to show it fails.
# Quick perceptron (reusing the idea from the previous notebook).
w = np.zeros(2)
b = 0.0
y_signed = 2 * y - 1  # convert 0/1 to -1/+1 for perceptron
for _ in range(50):
    for x_i, y_i in zip(X, y_signed):
        if y_i * (np.dot(w, x_i) + b) <= 0:
            w += y_i * x_i
            b += y_i

plot_data(X, y, ax=axes[1], title="Best linear boundary")
xlim = axes[1].get_xlim()
if abs(w[1]) > 1e-12:
    xs = np.linspace(*xlim, 100)
    ys = -(w[0] * xs + b) / w[1]
    axes[1].plot(xs, ys, color="black", linewidth=2, label="perceptron line")
axes[1].set_xlim(*xlim)
axes[1].set_ylim(axes[0].get_ylim())

preds = np.where(X @ w + b >= 0, 1, -1)
acc = (preds == y_signed).mean()
axes[1].set_title(f"Best linear boundary (accuracy: {acc:.0%})")
axes[1].legend()

plt.tight_layout()
plt.show()

## MLP architecture

Our network has three layers:

```
  Input (2)  ──→  Hidden (h neurons)  ──→  Output (1)
   x1, x2          σ(z) activations        σ(z) → probability
```

**Weight matrices and biases:**

| Connection | Weights | Bias |
|:-----------|:--------|:-----|
| Input → Hidden | $W_1 \in \mathbb{R}^{2 \times h}$ | $b_1 \in \mathbb{R}^{1 \times h}$ |
| Hidden → Output | $W_2 \in \mathbb{R}^{h \times 1}$ | $b_2 \in \mathbb{R}^{1 \times 1}$ |

Each hidden neuron computes a weighted sum of the inputs and passes it through a nonlinear **activation function**. Without the nonlinearity, stacking layers would be no more powerful than a single layer — the composition of two linear functions is still linear:

$$X W_1 W_2 = X (W_1 W_2) = X W_{\text{combined}}$$

The activation function breaks this linearity and gives the network its power.

### The sigmoid activation

We use the **sigmoid** function as our nonlinearity:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Key properties:
- Output is always between 0 and 1 — natural for probabilities
- Smooth and differentiable everywhere — needed for gradient-based learning
- Its derivative has a beautifully simple form: $\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$

In [ ]:
def sigmoid(z):
    """Sigmoid activation: maps any real number to (0, 1)."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def sigmoid_derivative(a):
    """Derivative of sigmoid, given the *output* a = sigmoid(z)."""
    return a * (1.0 - a)


# Plot sigmoid and its derivative side by side.
z = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(z, sigmoid(z), color="steelblue", linewidth=2)
axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("$\\sigma(z)$")
axes[0].set_xlabel("$z$")
axes[0].set_ylabel("$\\sigma(z)$")

a = sigmoid(z)
axes[1].plot(z, sigmoid_derivative(a), color="tab:orange", linewidth=2)
axes[1].set_title("$\\sigma'(z) = \\sigma(z)(1 - \\sigma(z))$")
axes[1].set_xlabel("$z$")
axes[1].set_ylabel("$\\sigma'(z)$")

plt.tight_layout()
plt.show()

## Forward pass

The forward pass computes the network output step by step:

$$z_1 = X \, W_1 + b_1 \qquad \text{(hidden pre-activation)}$$

$$a_1 = \sigma(z_1) \qquad \text{(hidden activation)}$$

$$z_2 = a_1 \, W_2 + b_2 \qquad \text{(output pre-activation)}$$

$$\hat{y} = a_2 = \sigma(z_2) \qquad \text{(predicted probability)}$$

Each step is just a matrix multiplication, a bias addition, and the sigmoid function. We store all intermediate values ($z_1, a_1, z_2, a_2$) because backpropagation will need them.

## Loss function — mean squared error

To train the network we need a measure of how wrong its predictions are. We use **mean squared error (MSE)**:

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

Intuition:
- Each term $(\hat{y}_i - y_i)^2$ measures how far the prediction is from the true label
- Squaring makes all errors positive and penalizes large errors more than small ones
- The mean averages over all samples

MSE is the simplest regression loss and easy to reason about — it is just the average squared distance between predictions and targets.

In [ ]:
def mse_loss(y_true, y_pred):
    """Compute mean squared error loss.

    Parameters
    ----------
    y_true : ndarray of shape (N,)
        True labels (0 or 1).
    y_pred : ndarray of shape (N,)
        Predicted values (between 0 and 1).
    """
    return np.mean((y_pred - y_true) ** 2)

## Backpropagation — deriving the gradients

Backpropagation computes the gradient of the loss with respect to every weight in the network, layer by layer from output back to input, using the **chain rule**.

### Output layer

We start from the MSE loss and work backward. The derivative of the loss with respect to the output $a_2$ is:

$$\frac{\partial \mathcal{L}}{\partial a_2} = \frac{2}{N}(a_2 - y)$$

Since $a_2 = \sigma(z_2)$, we multiply by the sigmoid derivative to get the **error signal** $\delta_2$:

$$\delta_2 = \frac{\partial \mathcal{L}}{\partial z_2} = \frac{2}{N}(a_2 - y) \odot \sigma'(a_2) \qquad \text{shape: } (N, 1)$$

where $\sigma'(a_2) = a_2 \odot (1 - a_2)$.

The gradients for the output weights and bias are:

$$\frac{\partial \mathcal{L}}{\partial W_2} = a_1^T \, \delta_2 \qquad \frac{\partial \mathcal{L}}{\partial b_2} = \sum \delta_2$$

### Hidden layer

We propagate the error **backward** through the output weights and then through the hidden sigmoid:

$$\delta_1 = (\delta_2 \, W_2^T) \odot \sigma'(a_1) \qquad \text{shape: } (N, h)$$

where $\sigma'(a_1) = a_1 \odot (1 - a_1)$ and $\odot$ is element-wise multiplication.

The gradients for the hidden weights and bias are:

$$\frac{\partial \mathcal{L}}{\partial W_1} = X^T \, \delta_1 \qquad \frac{\partial \mathcal{L}}{\partial b_1} = \sum \delta_1$$

That's it — four gradients, one per parameter matrix. The **chain rule** is the only calculus we need.

> **Note:** Unlike binary cross-entropy, the sigmoid derivative does *not* cancel out with MSE. This means the gradient includes an extra $\sigma'$ factor, which can slow down learning when the sigmoid is saturated (outputs near 0 or 1). This is a known trade-off — MSE is simpler to understand, but converges more slowly than BCE with sigmoid.

## Putting it all together — the MLP class

The class below packages the forward pass, backward pass, and weight update into a clean interface.

**Weight initialization** uses *Xavier/Glorot* scaling: weights are drawn from a normal distribution with standard deviation $\sqrt{2 / (\text{fan\_in} + \text{fan\_out})}$. This keeps the signal from vanishing or exploding as it flows through the network.

In [ ]:
class MLP:
    """A two-layer (one hidden) multi-layer perceptron for binary classification.

    Architecture: Input(n_input) -> Hidden(n_hidden, sigmoid) -> Output(1, sigmoid)
    """

    def __init__(self, n_input=2, n_hidden=8, n_output=1, seed=42):
        rng = np.random.default_rng(seed)

        # Xavier initialization.
        self.W1 = rng.normal(0, np.sqrt(2 / (n_input + n_hidden)), (n_input, n_hidden))
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.normal(0, np.sqrt(2 / (n_hidden + n_output)), (n_hidden, n_output))
        self.b2 = np.zeros((1, n_output))

    # ---- Forward pass ----

    def forward(self, X):
        """Compute predicted probabilities for input X."""
        self.z1 = X @ self.W1 + self.b1          # (N, n_hidden)
        self.a1 = sigmoid(self.z1)                # (N, n_hidden)
        self.z2 = self.a1 @ self.W2 + self.b2    # (N, 1)
        self.a2 = sigmoid(self.z2)                # (N, 1)
        return self.a2

    # ---- Backward pass (backpropagation) ----

    def backward(self, X, y_true):
        """Compute gradients for all weights and biases using MSE loss."""
        N = X.shape[0]
        y_true = y_true.reshape(-1, 1)

        # Output layer error signal (MSE gradient through sigmoid).
        # dL/da2 = 2/N * (a2 - y), then multiply by sigmoid derivative.
        delta2 = (2 / N) * (self.a2 - y_true) * sigmoid_derivative(self.a2)  # (N, 1)

        dW2 = self.a1.T @ delta2                            # (n_hidden, 1)
        db2 = np.sum(delta2, axis=0, keepdims=True)         # (1, 1)

        # Hidden layer error signal: propagate delta2 backward.
        delta1 = (delta2 @ self.W2.T) * sigmoid_derivative(self.a1)  # (N, n_hidden)

        dW1 = X.T @ delta1                                  # (n_input, n_hidden)
        db1 = np.sum(delta1, axis=0, keepdims=True)         # (1, n_hidden)

        return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

    # ---- Weight update (gradient descent) ----

    def update(self, grads, lr):
        """Apply one step of gradient descent."""
        self.W1 -= lr * grads["dW1"]
        self.b1 -= lr * grads["db1"]
        self.W2 -= lr * grads["dW2"]
        self.b2 -= lr * grads["db2"]

    # ---- Convenience ----

    def predict(self, X):
        """Return class labels (0 or 1) by thresholding at 0.5."""
        return (self.forward(X).ravel() >= 0.5).astype(int)

    def get_params(self):
        """Return a copy of all parameters (for snapshots)."""
        return (self.W1.copy(), self.b1.copy(), self.W2.copy(), self.b2.copy())

    def set_params(self, params):
        """Load parameters from a snapshot."""
        self.W1, self.b1, self.W2, self.b2 = params

In [ ]:
# Quick sanity check: forward pass on untrained network.
mlp = MLP(n_input=2, n_hidden=8, seed=42)
y_pred = mlp.forward(X)

print(f"Output shape: {y_pred.shape}")
print(f"First 5 predictions: {y_pred.ravel()[:5].round(3)}")
print(f"All outputs in (0, 1): {(y_pred > 0).all() and (y_pred < 1).all()}")
print(f"Initial loss: {mse_loss(y, y_pred.ravel()):.4f}")
print(f"Initial accuracy: {(mlp.predict(X) == y).mean():.0%} (random — expected ~50%)")

## Training

We use **full-batch gradient descent**: each epoch processes the entire dataset, computes the average gradient, and updates the weights once.

```
for each epoch:
    predictions = forward(X)          # forward pass
    loss        = MSE(y, predictions) # measure error
    gradients   = backward(X, y)      # compute gradients (backprop)
    update weights using gradients    # gradient descent step
```

We save **snapshots** of the weights at selected epochs so we can visualize how the decision boundary evolves.

In [ ]:
def train_mlp(mlp, X, y, lr=1.0, epochs=2000, snapshot_at=None):
    """Train the MLP with full-batch gradient descent.

    Parameters
    ----------
    mlp : MLP
        The network to train (modified in place).
    X : ndarray of shape (N, 2)
        Training inputs.
    y : ndarray of shape (N,)
        Training labels (0 or 1).
    lr : float
        Learning rate for gradient descent.
    epochs : int
        Number of training epochs.
    snapshot_at : list of int, optional
        Epochs at which to save a copy of the weights.

    Returns
    -------
    loss_history : list of float
    snapshots : dict mapping epoch -> parameter tuple
    """
    if snapshot_at is None:
        snapshot_at = []
    snapshot_at = set(snapshot_at)

    loss_history = []
    snapshots = {}

    for epoch in range(epochs):
        # Forward pass.
        y_pred = mlp.forward(X)

        # Compute loss.
        loss = mse_loss(y, y_pred.ravel())
        loss_history.append(loss)

        # Save snapshot before updating (so epoch 0 = initial state).
        if epoch in snapshot_at:
            snapshots[epoch] = mlp.get_params()

        # Backward pass.
        grads = mlp.backward(X, y)

        # Update weights.
        mlp.update(grads, lr)

    # Final snapshot.
    snapshots[epochs] = mlp.get_params()

    return loss_history, snapshots

In [ ]:
# Train the network.
# MSE + sigmoid converges slower than BCE, so we use more epochs and a higher
# learning rate to compensate.
mlp = MLP(n_input=2, n_hidden=8, seed=42)

snapshot_epochs = [0, 100, 500, 1500, 3000]
n_epochs = 5000

loss_history, snapshots = train_mlp(
    mlp, X, y,
    lr=5.0,
    epochs=n_epochs,
    snapshot_at=snapshot_epochs,
)

final_acc = (mlp.predict(X) == y).mean()
print(f"Final loss:     {loss_history[-1]:.4f}")
print(f"Final accuracy: {final_acc:.0%}")

### Loss curve

A healthy training run shows the loss decreasing smoothly and eventually levelling off.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, color="steelblue", linewidth=1.5)
ax.axhline(loss_history[-1], color="gray", linestyle="--", linewidth=1,
           label=f"final loss = {loss_history[-1]:.4f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (MSE)")
ax.set_title("Training loss")
ax.legend()
plt.tight_layout()
plt.show()

## Decision boundary

To visualize what the network has learned, we evaluate it on a fine grid of points covering the input space and color regions by predicted class. The boundary between the colored regions is the **decision boundary** — the curve where the network output is exactly 0.5.

In [ ]:
# Background colormap: light orange for class 0, light blue for class 1.
BG_CMAP = ListedColormap(["#fdd8b5", "#b5d4f5"])


def plot_decision_boundary(mlp, X, y, ax=None, title="", resolution=200):
    """Plot the network's decision boundary as a filled contour.

    Parameters
    ----------
    mlp : MLP
        Trained (or partially trained) network.
    X : ndarray of shape (N, 2)
    y : ndarray of shape (N,)
    ax : matplotlib Axes, optional
    title : str
    resolution : int
        Grid resolution for the contour.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    margin = 0.5
    x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
    y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, resolution),
        np.linspace(y_min, y_max, resolution),
    )
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    zz = mlp.forward(grid).reshape(xx.shape)

    ax.contourf(xx, yy, zz, levels=[0, 0.5, 1], cmap=BG_CMAP, alpha=0.6)
    ax.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=1.5)

    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=30, color="tab:orange",
               edgecolors="k", linewidths=0.3)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=30, color="tab:blue",
               edgecolors="k", linewidths=0.3)

    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_aspect("equal")
    if title:
        ax.set_title(title)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
plot_decision_boundary(mlp, X, y, ax=ax,
                       title=f"Trained MLP (accuracy: {final_acc:.0%})")
plt.show()

### How the boundary evolves during training

The snapshots below show the decision boundary at different stages of training — from the random initialization to the final trained state.

In [ ]:
all_snapshot_epochs = sorted(snapshots.keys())

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

temp_mlp = MLP(n_input=2, n_hidden=8)  # temporary MLP for loading snapshots

for ax, epoch in zip(axes, all_snapshot_epochs):
    temp_mlp.set_params(snapshots[epoch])
    acc = (temp_mlp.predict(X) == y).mean()
    plot_decision_boundary(temp_mlp, X, y, ax=ax,
                           title=f"Epoch {epoch} (acc: {acc:.0%})")

for ax in axes[len(all_snapshot_epochs):]:
    ax.axis("off")

plt.suptitle("Decision boundary evolution", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## What do the hidden neurons learn?

Each hidden neuron computes $\sigma(w_1 x_1 + w_2 x_2 + b)$ — this is essentially a **soft linear boundary** in the input space. The output neuron then combines these individual boundaries into the final curved decision region.

The plots below show the activation of each hidden neuron across the input space. Bright = high activation, dark = low activation. Notice how each neuron "carves out" a different half of the space.

In [ ]:
n_hidden = mlp.W1.shape[1]
n_cols = 4
n_rows = (n_hidden + n_cols - 1) // n_cols

margin = 0.5
x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
grid = np.column_stack([xx.ravel(), yy.ravel()])

# Compute hidden layer activations on the grid.
hidden_activations = sigmoid(grid @ mlp.W1 + mlp.b1)  # (grid_points, n_hidden)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
axes = axes.ravel()

for j in range(n_hidden):
    ax = axes[j]
    act = hidden_activations[:, j].reshape(xx.shape)
    im = ax.contourf(xx, yy, act, levels=20, cmap="RdYlBu_r", alpha=0.8)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=10, color="tab:orange",
               edgecolors="k", linewidths=0.2)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=10, color="tab:blue",
               edgecolors="k", linewidths=0.2)
    ax.set_title(f"Hidden neuron {j + 1}")
    ax.set_aspect("equal")
    plt.colorbar(im, ax=ax, fraction=0.046)

for ax in axes[n_hidden:]:
    ax.axis("off")

plt.suptitle("Individual hidden neuron activations", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Experiment 1: Learning rate

The **learning rate** controls how big each gradient descent step is:
- **Too small** → the network learns very slowly
- **Too large** → the updates overshoot and the loss may oscillate or diverge
- **Just right** → smooth, fast convergence

Let's train the same network with different learning rates and compare.

In [ ]:
learning_rates = [0.5, 2.0, 5.0, 20.0]
lr_results = {}

for lr in learning_rates:
    net = MLP(n_input=2, n_hidden=8, seed=42)  # same initialization each time
    losses, _ = train_mlp(net, X, y, lr=lr, epochs=2000)
    acc = (net.predict(X) == y).mean()
    lr_results[lr] = {"losses": losses, "mlp": net, "acc": acc}

# Loss curves.
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["tab:blue", "tab:green", "steelblue", "tab:red"]
for (lr, res), color in zip(lr_results.items(), colors):
    ax.plot(res["losses"], label=f"lr = {lr} (acc: {res['acc']:.0%})",
            color=color, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (MSE)")
ax.set_title("Training loss for different learning rates")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (lr, res) in zip(axes, lr_results.items()):
    plot_decision_boundary(res["mlp"], X, y, ax=ax,
                           title=f"lr = {lr} (acc: {res['acc']:.0%})")
plt.suptitle("Decision boundaries for different learning rates", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Experiment 2: Hidden layer size

The number of hidden neurons controls the network's **capacity** — how complex a decision boundary it can represent:
- **Too few** neurons → the boundary is too simple (underfitting)
- **Too many** neurons → the network has more capacity than needed (can overfit on noisy data, though our clean circles dataset is forgiving)

Pay special attention to `n_hidden = 1`: with a single hidden neuron, the network is essentially linear and cannot solve the circles problem.

In [ ]:
hidden_sizes = [1, 2, 4, 8, 16]
size_results = {}

for h in hidden_sizes:
    net = MLP(n_input=2, n_hidden=h, seed=42)
    losses, _ = train_mlp(net, X, y, lr=5.0, epochs=3000)
    acc = (net.predict(X) == y).mean()
    size_results[h] = {"losses": losses, "mlp": net, "acc": acc}

# Loss curves.
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["tab:red", "tab:orange", "tab:green", "steelblue", "tab:purple"]
for (h, res), color in zip(size_results.items(), colors):
    ax.plot(res["losses"], label=f"h = {h} (acc: {res['acc']:.0%})",
            color=color, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (MSE)")
ax.set_title("Training loss for different hidden layer sizes")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for ax, (h, res) in zip(axes, size_results.items()):
    plot_decision_boundary(res["mlp"], X, y, ax=ax,
                           title=f"h = {h} (acc: {res['acc']:.0%})")
plt.suptitle("Decision boundaries for different hidden layer sizes", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Key takeaways

- A **single perceptron** can only learn linear boundaries — it fails on problems like concentric circles.
- An **MLP** with one hidden layer and nonlinear activations can learn curved decision boundaries.
- **Backpropagation** computes gradients efficiently by applying the chain rule layer by layer, from output to input.
- With **MSE loss**, the sigmoid derivative appears explicitly in the gradient — this is simpler to understand but converges slower than binary cross-entropy (where the sigmoid derivative cancels out).
- The **learning rate** controls the trade-off between speed and stability of training.
- The **hidden layer size** controls model capacity — too small underfits, too large risks overfitting.
- Each **hidden neuron** learns a soft linear boundary; the output neuron combines them into a complex shape.

### What's next?

- **Binary cross-entropy loss** — faster convergence because the sigmoid derivative cancels in the gradient
- **ReLU activation** — faster training, avoids the vanishing gradient problem of sigmoid
- **Multiple hidden layers** — deeper networks can learn hierarchical features
- **Mini-batch SGD** — more efficient training on large datasets
- **Regularization** (dropout, weight decay) — prevents overfitting on noisy or small datasets